In [ ]:
# ==============================
# PHASE 05 - STEP 01
# Load Locked Test Data & Frozen Models
# ==============================

import os
import numpy as np
import tensorflow as tf
from google.colab import drive

# 1. MOUNT DRIVE
drive.mount('/content/drive')

# 2. PATHS (Strictly matching Phase 4 Output)
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"
TEST_PATH  = os.path.join(BASE_PATH, "Data", "Test_Set_Locked")
MODEL_PATH = os.path.join(BASE_PATH, "Models")

# 3. DEFINE FILES (Locked Set)
X_TEST_PATH = os.path.join(TEST_PATH, "X_test_locked.npy")
Y_TEST_PATH = os.path.join(TEST_PATH, "y_test_locked.npy")
G_TEST_PATH = os.path.join(TEST_PATH, "groups_test_locked.npy")

BASELINE_MODEL_PATH = os.path.join(MODEL_PATH, "model_baseline.keras")
ENHANCED_MODEL_PATH = os.path.join(MODEL_PATH, "model_enhanced.keras")

print("🚀 PHASE 05: VIDEO-LEVEL EVALUATION STARTED...\n")

# 4. VERIFY FILES EXIST (No Repair - Just Check)
required_files = [X_TEST_PATH, Y_TEST_PATH, G_TEST_PATH, BASELINE_MODEL_PATH, ENHANCED_MODEL_PATH]
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    raise FileNotFoundError(f"❌ STOP! Missing Phase 4 files: {missing_files}\n   Go back to Phase 4 and run it properly.")

print("✔ All Locked Files Found.")

# 5. LOAD DATA (Read-Only)
print("\n[Loading Locked Test Set...]")
X_test = np.load(X_TEST_PATH)
y_test = np.load(Y_TEST_PATH)
groups_test = np.load(G_TEST_PATH)

print(f"   ✅ X_test_locked Loaded: {X_test.shape}")
print(f"   ✅ y_test_locked Loaded: {y_test.shape}")
print(f"   ✅ groups_test_locked Loaded: {groups_test.shape}")

# 6. LOAD MODELS (Frozen)
print("\n[Loading Frozen Models...]")
try:
    baseline_model = tf.keras.models.load_model(BASELINE_MODEL_PATH)
    print("   ✅ Baseline Model Loaded.")

    enhanced_model = tf.keras.models.load_model(ENHANCED_MODEL_PATH)
    print("   ✅ Enhanced Model (v1.0) Loaded.")
except Exception as e:
    raise RuntimeError(f"❌ Error loading models: {e}")

# 7. SAFETY CHECK
if len(X_test) != len(groups_test):
    raise ValueError("❌ CRITICAL: X_test and Groups length mismatch!")

print("\n------------------------------------------------")
print("✅ PHASE 05 – STEP 01 COMPLETE")
print("Locked Data & Models Ready.")

In [ ]:
# ==============================
# PHASE 05 - STEP 02
# Prediction + Video-Level Majority Voting
# ==============================

import os
import numpy as np
import pandas as pd
from collections import Counter

print("🚀 PHASE 05 – STEP 02 STARTED")

# --------------------------------------------------
# Assumes STEP-01 already ran and loaded:
# X_test, y_test, groups_test
# baseline_model, enhanced_model
# BASE_PATH
# --------------------------------------------------

# 1. SEQUENCE-LEVEL PREDICTION (INFERENCE ONLY)
print("[1/3] Running sequence-level predictions...")

baseline_probs = baseline_model.predict(X_test, verbose=1)
enhanced_probs = enhanced_model.predict(X_test, verbose=1)

baseline_seq_preds = np.argmax(baseline_probs, axis=1)
enhanced_seq_preds = np.argmax(enhanced_probs, axis=1)

print("✔ Sequence predictions complete.")

# 2. VIDEO-LEVEL MAJORITY VOTING
print("\n[2/3] Applying video-level majority voting...")

video_results = []

unique_videos = np.unique(groups_test)

for vid in unique_videos:
    idx = np.where(groups_test == vid)[0]

    y_true = y_test[idx][0]  # one label per video (guaranteed)
    vote_base = Counter(baseline_seq_preds[idx]).most_common(1)[0][0]
    vote_enh  = Counter(enhanced_seq_preds[idx]).most_common(1)[0][0]

    video_results.append({
        "video_id": vid,
        "y_true": y_true,
        "y_pred_baseline": vote_base,
        "y_pred_enhanced": vote_enh
    })

df_video = pd.DataFrame(video_results)

print(f"✔ Voting complete. Total videos: {len(df_video)}")

# 3. SAVE VIDEO-LEVEL PREDICTIONS
print("\n[3/3] Saving video-level predictions...")

SAVE_DIR = os.path.join(BASE_PATH, "Results", "Predictions")
os.makedirs(SAVE_DIR, exist_ok=True)

save_path = os.path.join(SAVE_DIR, "final_video_predictions.csv")
df_video.to_csv(save_path, index=False)

print(f"✔ Saved: {save_path}")

print("\n✅ PHASE 05 – STEP 02 COMPLETE")
print("Ready for STEP 05-03 (Comparison & Selection)")

In [ ]:
# ==========================================
# PHASE 05 - STEP 03
# Comparison & Final Model Selection
# ==========================================

import os
import pandas as pd
from sklearn.metrics import accuracy_score

print("🚀 STEP 03: MODEL COMPARISON STARTED...\n")

# 1. PATHS
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"
PRED_PATH = os.path.join(BASE_PATH, "Results", "Predictions")
METRIC_PATH = os.path.join(BASE_PATH, "Results", "Metrics")

os.makedirs(METRIC_PATH, exist_ok=True)

# 2. LOAD VIDEO-LEVEL RESULTS
load_path = os.path.join(PRED_PATH, "final_video_predictions.csv")
if not os.path.exists(load_path):
    raise FileNotFoundError("❌ Missing predictions file. Did you run Step 02?")

df = pd.read_csv(load_path)
print(f"✔ Loaded predictions for {len(df)} videos.")

# 3. COMPUTE VIDEO-LEVEL ACCURACY
acc_baseline = accuracy_score(df["y_true"], df["y_pred_baseline"])
acc_enhanced = accuracy_score(df["y_true"], df["y_pred_enhanced"])

# 4. BUILD COMPARISON TABLE
summary = pd.DataFrame({
    "Model": ["Baseline LSTM", "Enhanced Bi-LSTM"],
    "Video-Level Accuracy": [acc_baseline, acc_enhanced]
})

# 5. SAVE
out_path = os.path.join(METRIC_PATH, "phase05_model_comparison.csv")
summary.to_csv(out_path, index=False)

# 6. DISPLAY
print("\n📊 FINAL SCORECARD (Video-Level Accuracy):")
print(summary)

# 7. SELECT WINNER
winner = "Enhanced Bi-LSTM" if acc_enhanced >= acc_baseline else "Baseline LSTM"
print(f"\n🏆 Selected Final Model (v1.0): {winner}")
print(f"   Saved → {out_path}")

print("\n✅ PHASE 05 – STEP 03 COMPLETE")
print("👉 Proceed to STEP 05-04 (Confusion Matrix & Final Metrics)")

In [ ]:
# ==========================================
# PHASE 05 - STEP 04 (FINAL BEST VERSION)
# Confusion Matrix & Classification Report
# (Dynamic winner-based selection)
# ==========================================

import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

print("🚀 PHASE 05 - STEP 04 STARTED...\n")

# 1. PATHS
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"
PRED_PATH = os.path.join(BASE_PATH, "Results", "Predictions")
FIG_PATH = os.path.join(BASE_PATH, "Results", "Figures")
METRIC_PATH = os.path.join(BASE_PATH, "Results", "Metrics")

os.makedirs(FIG_PATH, exist_ok=True)
os.makedirs(METRIC_PATH, exist_ok=True)

# 2. LOAD VIDEO-LEVEL RESULTS
pred_path = os.path.join(PRED_PATH, "final_video_predictions.csv")
if not os.path.exists(pred_path):
    raise FileNotFoundError("❌ Missing predictions file. Run Step 02 first.")

df = pd.read_csv(pred_path)

# 3. COMPUTE ACCURACY AGAIN (SAFE & SMART)
# This calculates who is the real winner right now.
acc_baseline = accuracy_score(df["y_true"], df["y_pred_baseline"])
acc_enhanced = accuracy_score(df["y_true"], df["y_pred_enhanced"])

# 4. SELECT FINAL MODEL DYNAMICALLY
if acc_enhanced >= acc_baseline:
    final_model_name = "Enhanced Bi-LSTM"
    y_pred = df["y_pred_enhanced"]
    print(f"🏆 Auto-Selected Winner: {final_model_name} (Acc: {acc_enhanced:.2%})")
else:
    final_model_name = "Baseline LSTM"
    y_pred = df["y_pred_baseline"]
    print(f"🏆 Auto-Selected Winner: {final_model_name} (Acc: {acc_baseline:.2%})")

y_true = df["y_true"]
CLASS_NAMES = ["NoFall", "Faint", "Slip", "Trip"]

# 5. CONFUSION MATRIX
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title(f"Video-Level Confusion Matrix (Final Model: {final_model_name})")

cm_path = os.path.join(FIG_PATH, "phase05_confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

# 6. CLASSIFICATION REPORT
report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()
report_path = os.path.join(METRIC_PATH, "phase05_classification_report.csv")
report_df.to_csv(report_path)

print(f"✔ Confusion Matrix saved → {cm_path}")
print(f"✔ Classification Report saved → {report_path}")

# 7. PRINT RESULTS (So you can see F1 Scores NOW)
print(f"\n📊 FINAL DETAILED REPORT FOR: {final_model_name}")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

print("\n✅ PHASE 05 COMPLETE")